# SHIPP Stage 5A — Silver Listings Development

**Method:** Spark Window Current-State Reconstruction

```text
Bronze listing history
    ↓
partition by listing_id
    ↓
order by _pg_lsn DESC, _sort_by DESC
    ↓
select rn = 1
    ↓
retain current-row change types
    ↓
quality checks
    ↓
silver_listings
```

This notebook is for development and validation before promoting logic to
`data_pipeline/silver/silver_listings.py`.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
CATALOG = "bootcamp_students"

BRONZE_SCHEMA = "shipp_bronze"
SILVER_SCHEMA = "shipp_silver"

SOURCE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.lb_listings_history"
TARGET_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA}"
TARGET_TABLE = f"{TARGET_SCHEMA}.silver_listings"

BUSINESS_KEY = "listing_id"

VALID_CHANGE_TYPES = {
    "insert",
    "update_postimage",
}

VALID_LISTING_STATUSES = {
    "DRAFT",
    "AVAILABLE",
    "UNAVAILABLE",
    "WITHDRAWN",
    "EXPIRED",
}

print("SOURCE_TABLE =", SOURCE_TABLE)
print("TARGET_TABLE =", TARGET_TABLE)
print("BUSINESS_KEY =", BUSINESS_KEY)

In [ ]:
# STEP 3
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}
""")

print(f"Target schema ready: {TARGET_SCHEMA}")

In [ ]:
# STEP 4
bronze_df = spark.table(SOURCE_TABLE)

print("Bronze loaded successfully.")
print("Bronze row count:", bronze_df.count())

bronze_df.printSchema()

In [ ]:
#STEP 5
REQUIRED_SOURCE_COLUMNS = {
    "listing_id",
    "donor_id",
    "title",
    "description",
    "category",
    "condition",
    "latitude",
    "longitude",
    "available_from",
    "available_until",
    "status",
    "updated_at",
    "_pg_change_type",
    "_pg_lsn",
    "_sort_by",
}

actual_columns = set(bronze_df.columns)

missing_columns = REQUIRED_SOURCE_COLUMNS - actual_columns

print("Missing required columns:", missing_columns)

assert not missing_columns, (
    f"Bronze schema contract failed. "
    f"Missing columns: {sorted(missing_columns)}"
)

print("PASS — Bronze source schema contract.")

In [ ]:
# STEP 6
display(
    bronze_df
    .groupBy("_pg_change_type")
    .count()
    .orderBy("_pg_change_type")
)

In [ ]:
# STEP 7

usable_events_df = (
    bronze_df
    .filter(
        F.col("_pg_change_type").isin(
            list(VALID_CHANGE_TYPES)
        )
    )
)

print("Usable CDC rows:", usable_events_df.count())

display(
    usable_events_df
    .select(
        "listing_id",
        "_pg_change_type",
        "_pg_lsn",
        "_sort_by",
        "status",
        "updated_at",
    )
    .orderBy(
        "listing_id",
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
)

In [ ]:
# STEP 8 — Create the Window and rank events
latest_window = (
    Window
    .partitionBy(BUSINESS_KEY)
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
)

ranked_df = (
    usable_events_df
    .withColumn(
        "_current_rn",
        F.row_number().over(latest_window)
    )
)

display(
    ranked_df
    .select(
        "listing_id",
        "_pg_change_type",
        "_pg_lsn",
        "_sort_by",
        "status",
        "_current_rn",
    )
    .orderBy(
        "listing_id",
        "_current_rn",
    )
)

In [ ]:
# STEP 9 — Select exactly one newest row per listing_id
latest_events_df = (
    ranked_df
    .filter(F.col("_current_rn") == 1)
)

print("Current listing rows:", latest_events_df.count())

display(
    latest_events_df
    .select(
        "listing_id",
        "title",
        "status",
        "category",
        "_pg_change_type",
        "_pg_lsn",
        "_sort_by",
    )
    .orderBy("listing_id")
)

In [ ]:
# STEP 10
silver_df = (
    latest_events_df
    .select(
        # Business columns
        F.col("listing_id"),
        F.col("donor_id"),
        F.col("title"),
        F.col("description"),
        F.col("category"),
        F.col("condition"),
        F.col("latitude"),
        F.col("longitude"),
        F.col("available_from"),
        F.col("available_until"),
        F.col("status"),
        F.col("updated_at"),

        # Traceability columns
        F.col("_pg_lsn").alias("source_pg_lsn"),
        F.col("_sort_by").alias("source_sort_by"),
        F.col("_pg_change_type").alias("source_change_type"),

        # Silver processing metadata
        F.current_timestamp().alias("silver_processed_at"),
    )
)

print("Silver candidate rows:", silver_df.count())

silver_df.printSchema()

display(
    silver_df.orderBy("listing_id")
)

In [ ]:
# ------------------------------------------------------------
# Silver Listings — Pre-write Quality Gate STEP 11 
# ------------------------------------------------------------

row_count = silver_df.count()

duplicate_listing_ids = (
    silver_df
    .groupBy(BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_listing_ids = (
    silver_df
    .filter(F.col("listing_id").isNull())
    .count()
)

null_donor_ids = (
    silver_df
    .filter(F.col("donor_id").isNull())
    .count()
)

null_categories = (
    silver_df
    .filter(F.col("category").isNull())
    .count()
)

invalid_latitudes = (
    silver_df
    .filter(
        F.col("latitude").isNotNull()
        & ~F.col("latitude").between(-90, 90)
    )
    .count()
)

invalid_longitudes = (
    silver_df
    .filter(
        F.col("longitude").isNotNull()
        & ~F.col("longitude").between(-180, 180)
    )
    .count()
)

invalid_statuses = (
    silver_df
    .filter(
        F.col("status").isNull()
        | ~F.col("status").isin(list(VALID_LISTING_STATUSES))
    )
    .count()
)

invalid_availability_ranges = (
    silver_df
    .filter(
        F.col("available_from").isNotNull()
        & F.col("available_until").isNotNull()
        & (F.col("available_until") < F.col("available_from"))
    )
    .count()
)

invalid_source_change_types = (
    silver_df
    .filter(
        ~F.col("source_change_type").isin(
            list(VALID_CHANGE_TYPES)
        )
    )
    .count()
)

print("Silver row count:", row_count)
print("Duplicate listing_id violations:", duplicate_listing_ids)
print("Null listing_id violations:", null_listing_ids)
print("Null donor_id violations:", null_donor_ids)
print("Null category violations:", null_categories)
print("Invalid latitude violations:", invalid_latitudes)
print("Invalid longitude violations:", invalid_longitudes)
print("Invalid status violations:", invalid_statuses)
print("Invalid availability-range violations:", invalid_availability_ranges)
print("Invalid source change-type violations:", invalid_source_change_types)

assert row_count > 0, "QUALITY FAILED: Silver contains zero rows."
assert duplicate_listing_ids == 0, "QUALITY FAILED: Duplicate listing_id."
assert null_listing_ids == 0, "QUALITY FAILED: Null listing_id."
assert null_donor_ids == 0, "QUALITY FAILED: Null donor_id."
assert null_categories == 0, "QUALITY FAILED: Null category."
assert invalid_latitudes == 0, "QUALITY FAILED: Invalid latitude."
assert invalid_longitudes == 0, "QUALITY FAILED: Invalid longitude."
assert invalid_statuses == 0, "QUALITY FAILED: Invalid listing status."
assert invalid_availability_ranges == 0, "QUALITY FAILED: Invalid availability range."
assert invalid_source_change_types == 0, "QUALITY FAILED: Invalid source change type."

print("PASS — Silver Listings pre-write quality gate.")

In [ ]:
# step 12 
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print("Silver table written successfully.")
print("TARGET_TABLE =", TARGET_TABLE)


written_silver_df = spark.table(TARGET_TABLE)

written_row_count = written_silver_df.count()

print("Written Silver row count:", written_row_count)

assert written_row_count == silver_df.count(), (
    "WRITE VALIDATION FAILED: "
    "Written row count does not match validated Silver DataFrame."
)

print("PASS — Silver table write confirmed.")

In [ ]:
# step 15 UNDER STEP 12
run2_rows = [
    row.asDict(recursive=True)
    for row in (
        spark.table(TARGET_TABLE)
        .select(*logical_columns)
        .orderBy(BUSINESS_KEY)
        .collect()
    )
]

run2_row_count = len(run2_rows)

print("RUN 1 logical row count:", run1_row_count)
print("RUN 2 logical row count:", run2_row_count)

assert run1_row_count == run2_row_count, (
    "IDEMPOTENCY FAILED: row count changed after rerun."
)

assert run1_rows == run2_rows, (
    "IDEMPOTENCY FAILED: logical Silver contents changed after rerun."
)

print("PASS — Silver Listings rerun is logically idempotent.")

In [ ]:
#step 13
silver_table_df = spark.table(TARGET_TABLE)

EXPECTED_SILVER_COLUMNS = [
    "listing_id",
    "donor_id",
    "title",
    "description",
    "category",
    "condition",
    "latitude",
    "longitude",
    "available_from",
    "available_until",
    "status",
    "updated_at",
    "source_pg_lsn",
    "source_sort_by",
    "source_change_type",
    "silver_processed_at",
]

actual_silver_columns = silver_table_df.columns

print("Expected columns:", EXPECTED_SILVER_COLUMNS)
print("Actual columns:  ", actual_silver_columns)

assert actual_silver_columns == EXPECTED_SILVER_COLUMNS, (
    "SCHEMA VALIDATION FAILED."
)

print("PASS — Silver target schema is correct.")

demo_listing_df = (
    silver_table_df
    .filter(F.col("listing_id") == "demo-listing-001")
)

demo_listing_count = demo_listing_df.count()

print("demo-listing-001 row count:", demo_listing_count)

display(
    demo_listing_df.select(
        "listing_id",
        "donor_id",
        "title",
        "category",
        "status",
        "latitude",
        "longitude",
        "source_pg_lsn",
        "source_sort_by",
        "source_change_type",
    )
)

In [ ]:
# ------------------------------------------------------------
# 1. Duplicate-key validation -- STEP 14
# ------------------------------------------------------------

duplicate_keys_df = (
    silver_table_df
    .groupBy("listing_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_key_count = duplicate_keys_df.count()

print("Duplicate listing_id count:", duplicate_key_count)

assert duplicate_key_count == 0, (
    "VALIDATION FAILED: duplicate listing_id found."
)

print("PASS — No duplicate listing_id.")

# ------------------------------------------------------------
# 2. Compare persisted Silver source metadata
#    with the latest usable Bronze event.
# ------------------------------------------------------------

expected_source_df = (
    latest_events_df
    .select(
        "listing_id",
        F.col("_pg_lsn").alias("expected_pg_lsn"),
        F.col("_sort_by").alias("expected_sort_by"),
        F.col("_pg_change_type").alias("expected_change_type"),
    )
)

source_comparison_df = (
    silver_table_df.alias("s")
    .join(
        expected_source_df.alias("b"),
        on="listing_id",
        how="inner",
    )
    .select(
        "listing_id",
        "source_pg_lsn",
        "expected_pg_lsn",
        "source_sort_by",
        "expected_sort_by",
        "source_change_type",
        "expected_change_type",
    )
)

display(
    source_comparison_df.orderBy("listing_id")
)

source_mismatch_count = (
    source_comparison_df
    .filter(
        (F.col("source_pg_lsn") != F.col("expected_pg_lsn"))
        | (F.col("source_sort_by") != F.col("expected_sort_by"))
        | (F.col("source_change_type") != F.col("expected_change_type"))
    )
    .count()
)

print("Source metadata mismatches:", source_mismatch_count)

assert source_mismatch_count == 0, (
    "VALIDATION FAILED: Silver source metadata "
    "does not match latest Bronze events."
)

print("PASS — Silver rows trace correctly to latest Bronze events.")

In [ ]:
#STEP 15 

logical_columns = [
    column
    for column in spark.table(TARGET_TABLE).columns
    if column != "silver_processed_at"
]

run1_rows = [
    row.asDict(recursive=True)
    for row in (
        spark.table(TARGET_TABLE)
        .select(*logical_columns)
        .orderBy(BUSINESS_KEY)
        .collect()
    )
]

run1_row_count = len(run1_rows)

print("RUN 1 logical row count:", run1_row_count)

assert run1_row_count == 2, (
    f"Expected 2 Silver rows before rerun, found {run1_row_count}"
)

print("PASS — Run 1 logical state captured.")

## Validation gate

Do not write the target until these checks pass.


In [ ]:
row_count = current_listings.count()
duplicate_keys = (
    current_listings.groupBy("listing_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
null_keys = current_listings.filter(F.col("listing_id").isNull()).count()
invalid_coords = current_listings.filter(
    ~F.col("latitude").between(-90, 90)
    | ~F.col("longitude").between(-180, 180)
).count()

print("current rows:", row_count)
print("duplicate listing_id:", duplicate_keys)
print("null listing_id:", null_keys)
print("invalid coordinates:", invalid_coords)

assert duplicate_keys == 0
assert null_keys == 0
assert invalid_coords == 0


In [ ]:
display(
    current_listings.filter(
        F.col("listing_id").isin(
            "demo-listing-001",
            "demo-listing-002",
        )
    )
)


## Write only after validation passes

The overwrite is intentionally idempotent for the capstone current-state table.


In [ ]:
(
    current_listings.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET)
)

print("Wrote:", TARGET)
print("Silver listing rows:", spark.table(TARGET).count())
